# Clean the HCP by care level and LGA table from Excel (ML-ready long format)

This notebook builds a simple data pipeline for the uploaded Excel file and reshapes the care level columns into an ML-ready long table.

## Cleaning rules
- read only the required worksheet
- detect the real header row from the Excel sheet
- keep only the required columns
- drop completely empty rows
- trim and normalize whitespace in text fields
- convert blank strings to missing values
- rename the columns for psql / analysis
- remove commas from numeric fields and convert to numbers
- reshape `Level 1` to `Level 4` into a single `care_level` column
- create a `people_count` column for ML-ready long format
- keep the total column only for validation in the wide cleaned dataframe

In [20]:
from pathlib import Path
import pandas as pd

INPUT_XLSX = Path("input/gen_data_home_care_demand_by_recipient_loc.xlsx")
SHEET_NAME = "Table 2.1 (LGA)"

INPUT_XLSX, SHEET_NAME

(PosixPath('input/gen_data_home_care_demand_by_recipient_loc.xlsx'),
 'Table 2.1 (LGA)')

In [21]:
raw_df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET_NAME,
    header=None
)

raw_df.head(15)

,0,1,2,3,4,5,6
0,Table 2.1: Number of people using home care (H...,NaN,NaN,NaN,NaN,NaN,NaN
1,Return to contents,NaN,NaN,NaN,NaN,NaN,NaN
2,LGA (2016) code,LGA (2016) name,Level 1,Level 2,Level 3,Level 4,Total
3,10050,Albury (C),80,374,224,85,763
4,10130,Armidale Regional (A),12,128,101,54,295
5,10250,Ballina (A),121,568,386,178,1253
6,10300,Balranald (A),7,10,4,4,25
7,10470,Bathurst Regional (A),9,169,150,81,409
8,10550,Bega Valley (A),44,387,221,75,727
9,10600,Bellingen (A),9,71,90,31,201


In [22]:
def clean_hcp_by_lga_long(input_path: Path, sheet_name: str = "Table 2.1"):
    # Read only the required worksheet without assuming the header row position.
    df = pd.read_excel(input_path, sheet_name=sheet_name, header=None)

    # Find the real header row by matching the first two header cells.
    col0 = df.iloc[:, 0].astype("string").str.strip()
    col1 = df.iloc[:, 1].astype("string").str.strip()

    header_mask = (
        col0.str.contains(r"^LGA \(2016\) code$", na=False)
        & col1.str.contains(r"^LGA \(2016\) name$", na=False)
    )

    if not header_mask.any():
        raise ValueError("Could not find the header row for the HCP by LGA sheet.")

    header_row = header_mask[header_mask].index[0]

    # Rebuild the dataframe using the detected header row.
    df = df.iloc[header_row:].copy()
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

    # Keep only the required columns from the source sheet.
    required_source_columns = [
        "LGA (2016) code",
        "LGA (2016) name",
        "Level 1",
        "Level 2",
        "Level 3",
        "Level 4",
        "Total",
    ]

    missing_cols = [c for c in required_source_columns if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected columns: {missing_cols}")

    df = df[required_source_columns].copy()

    # Rename columns for analysis / psql.
    df = df.rename(
        columns={
            "LGA (2016) code": "lga_code_2016",
            "LGA (2016) name": "lga_name_2016",
            "Level 1": "level_1",
            "Level 2": "level_2",
            "Level 3": "level_3",
            "Level 4": "level_4",
            "Total": "total_people_using_home_care",
        }
    )

    # Normalize text columns.
    text_cols = ["lga_code_2016", "lga_name_2016"]
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "<NA>": pd.NA})

    # Clean numeric columns.
    numeric_cols = [
        "level_1",
        "level_2",
        "level_3",
        "level_4",
        "total_people_using_home_care",
    ]

    for col in numeric_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "<NA>": pd.NA})
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop rows that are completely empty after cleaning.
    df = df.dropna(how="all").reset_index(drop=True)

    # Drop rows without LGA code or LGA name.
    df = df.dropna(subset=["lga_code_2016", "lga_name_2016"]).reset_index(drop=True)

    # Optional safety filter: remove repeated header rows if they appear again inside the sheet.
    df = df[
        ~df["lga_code_2016"].astype("string").str.contains(r"^LGA \(2016\) code$", na=False)
    ].reset_index(drop=True)

    # Build ML-ready long format.
    long_df = df.melt(
        id_vars=["lga_code_2016", "lga_name_2016"],
        value_vars=["level_1", "level_2", "level_3", "level_4"],
        var_name="care_level_raw",
        value_name="people_count",
    )

    long_df["care_level"] = (
        long_df["care_level_raw"]
        .astype("string")
        .str.extract(r"(\d+)")
        .astype("Int64")
    )

    long_df = long_df.drop(columns=["care_level_raw"])

    # Final column order.
    long_df = long_df[
        ["lga_code_2016", "lga_name_2016", "care_level", "people_count"]
    ].copy()

    # Optional: drop rows where people_count is missing.
    long_df = long_df.dropna(subset=["people_count"]).reset_index(drop=True)

    summary = {
        "header_row_index": int(header_row),
        "wide_rows_after_cleaning": int(len(df)),
        "long_rows_after_melt": int(len(long_df)),
        "null_people_count_rows": int(long_df["people_count"].isna().sum()),
        "duplicate_long_rows": int(long_df.duplicated().sum()),
    }

    return df, long_df, summary

In [23]:
clean_wide_df, clean_long_df, summary = clean_hcp_by_lga_long(INPUT_XLSX, SHEET_NAME)
summary

{'header_row_index': 2,
 'wide_rows_after_cleaning': 522,
 'long_rows_after_melt': 2088,
 'null_people_count_rows': 0,
 'duplicate_long_rows': 0}

In [24]:
clean_long_df.head(12)

,lga_code_2016,lga_name_2016,care_level,people_count
0,10050,Albury (C),1,80
1,10130,Armidale Regional (A),1,12
2,10250,Ballina (A),1,121
3,10300,Balranald (A),1,7
4,10470,Bathurst Regional (A),1,9
5,10550,Bega Valley (A),1,44
6,10600,Bellingen (A),1,9
7,10650,Berrigan (A),1,23
8,10750,Blacktown (C),1,210
9,10800,Bland (A),1,4


In [25]:
print(f"Blank rows remaining in long_df: {int(clean_long_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining in long_df: {int(clean_long_df.duplicated().sum())}")
print(f"Null lga_code_2016 rows: {int(clean_long_df['lga_code_2016'].isna().sum())}")
print(f"Null lga_name_2016 rows: {int(clean_long_df['lga_name_2016'].isna().sum())}")
print(f"Null care_level rows: {int(clean_long_df['care_level'].isna().sum())}")
print(f"Null people_count rows: {int(clean_long_df['people_count'].isna().sum())}")
print(f"Data types:\n{clean_long_df.dtypes}")

Blank rows remaining in long_df: 0
Exact duplicates remaining in long_df: 0
Null lga_code_2016 rows: 0
Null lga_name_2016 rows: 0
Null care_level rows: 0
Null people_count rows: 0
Data types:
lga_code_2016    string
lga_name_2016    string
care_level        Int64
people_count      Int64
dtype: object


# Insert Clean Long Data into DB

In [26]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

print(f"Connected to database: {DB_NAME}")

Connected to database: carelink


In [27]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_long_df.copy()

target_columns = [
    "lga_code_2016",
    "lga_name_2016",
    "care_level",
    "people_count",
]

df_insert = df_insert[target_columns].copy()

text_cols = [
    "lga_code_2016",
    "lga_name_2016",
]

for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

df_insert["care_level"] = df_insert["care_level"].astype("Int64")
df_insert["people_count"] = pd.to_numeric(df_insert["people_count"], errors="coerce").astype("Int64")

df_insert.head()

,lga_code_2016,lga_name_2016,care_level,people_count
0,10050,Albury (C),1,80
1,10130,Armidale Regional (A),1,12
2,10250,Ballina (A),1,121
3,10300,Balranald (A),1,7
4,10470,Bathurst Regional (A),1,9


In [28]:
# -----------------------------
# 4. CREATE TABLE FROM SCHEMA
# -----------------------------
from pathlib import Path
from sqlalchemy import text

schema_path = Path("schemas/home_care_demand_by_recipient_lga.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")

Schema created or already exists.


In [29]:
# -----------------------------
# 5. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "home_care_demand_by_recipient_lga",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into home_care_demand_by_recipient_lga")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM home_care_demand_by_recipient_lga"))
    row_count = result.scalar()

print(f"Total rows now in home_care_demand_by_recipient_lga: {row_count}")

Inserted 2088 rows into home_care_demand_by_recipient_lga
Total rows now in home_care_demand_by_recipient_lga: 2088
